# TrustiPay Behavior Profile K-Means Notebook

This notebook trains and inspects the **K-Means behavior profiling model** used by the backend service.

- Uses the same core behavioral features as `app/services/behavior_profile.py`
- Fits `k=4` clusters
- Produces profile labels and centroid insights

In [ ]:
# %pip install -q pandas numpy scikit-learn matplotlib seaborn

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
K = 4

In [ ]:
candidates = [
    Path('../synthetic_data/TrustiPay Synthetic Transactions.csv'),
    Path('synthetic_data/TrustiPay Synthetic Transactions.csv'),
]

DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Could not find synthetic transaction CSV.')

df = pd.read_csv(DATA_PATH)
df['occurred_at'] = pd.to_datetime(df['occurred_at'], utc=True, errors='coerce')
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df = df.dropna(subset=['occurred_at', 'amount', 'direction', 'user_ref'])

df['direction'] = df['direction'].replace({'debit': 'expense', 'credit': 'income'})
df['category'] = df['category'].fillna('Unknown')
print(f'Loaded {len(df):,} rows from {DATA_PATH}')
df.head()

In [ ]:
# Build user-level behavioral features
income = df[df['direction'].str.lower() == 'income'].groupby('user_ref')['amount'].sum()
expense = df[df['direction'].str.lower() == 'expense'].groupby('user_ref')['amount'].sum()

non_essential_categories = {'shopping', 'entertainment'}
non_essential = (
    df[(df['direction'].str.lower() == 'expense') & (df['category'].str.lower().isin(non_essential_categories))]
    .groupby('user_ref')['amount'].sum()
)

expense_df = df[df['direction'].str.lower() == 'expense'].copy()
expense_df['year_week'] = expense_df['occurred_at'].dt.strftime('%G-%V')
weekly = expense_df.groupby(['user_ref', 'year_week'])['amount'].sum().reset_index()
stability = weekly.groupby('user_ref')['amount'].agg(['mean', 'std']).fillna(0.0)
stability['spending_stability'] = np.where(stability['mean'] > 0, stability['std'] / stability['mean'], 0.0)

anomaly_per_user = expense_df.groupby('user_ref')['amount'].apply(lambda s: ((s - s.mean()) / (s.std(ddof=0) + 1e-9) >= 2.5).sum())
tx_count = df.groupby('user_ref').size()

users = sorted(df['user_ref'].unique())
features = pd.DataFrame(index=users)
features['income_total'] = income.reindex(users).fillna(0.0)
features['expense_total'] = expense.reindex(users).fillna(0.0)
features['non_essential_total'] = non_essential.reindex(users).fillna(0.0)
features['spending_stability'] = stability['spending_stability'].reindex(users).fillna(0.5)
features['anomaly_count'] = anomaly_per_user.reindex(users).fillna(0.0)
features['tx_count'] = tx_count.reindex(users).fillna(0)

features['savings_ratio'] = np.where(
    features['income_total'] > 0,
    (features['income_total'] - features['expense_total']) / features['income_total'],
    0.0,
)
features['non_essential_ratio'] = np.where(
    features['expense_total'] > 0,
    features['non_essential_total'] / features['expense_total'],
    0.0,
)
features['anomaly_rate_per_100_tx'] = np.where(features['tx_count'] > 0, (features['anomaly_count'] / features['tx_count']) * 100, 0.0)

model_features = features[['savings_ratio', 'non_essential_ratio', 'spending_stability', 'anomaly_rate_per_100_tx']].copy()
model_features.describe().T

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(model_features)

kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

clustered = model_features.copy()
clustered['cluster_id'] = clusters
clustered.head()

In [ ]:
centroids = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=model_features.columns
)
centroids['cluster_id'] = centroids.index
display(centroids.round(4))

profile_map = {}
tmp = centroids.copy()
conservative = tmp.sort_values(['savings_ratio', 'non_essential_ratio', 'spending_stability'], ascending=[False, True, True]).iloc[0]['cluster_id']
profile_map[int(conservative)] = 'Conservative Saver'
tmp = tmp[tmp['cluster_id'] != conservative]
volatile = tmp.sort_values(['spending_stability', 'anomaly_rate_per_100_tx', 'savings_ratio'], ascending=[False, False, True]).iloc[0]['cluster_id']
profile_map[int(volatile)] = 'Volatile Risk User'
tmp = tmp[tmp['cluster_id'] != volatile]
lifestyle = tmp.sort_values(['non_essential_ratio', 'savings_ratio'], ascending=[False, True]).iloc[0]['cluster_id']
profile_map[int(lifestyle)] = 'Lifestyle Spender'
tmp = tmp[tmp['cluster_id'] != lifestyle]
profile_map[int(tmp.iloc[0]['cluster_id'])] = 'Balanced Spender'

clustered['profile'] = clustered['cluster_id'].map(profile_map)
clustered['profile'].value_counts()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=clustered.reset_index(),
    x='savings_ratio',
    y='non_essential_ratio',
    hue='profile',
    alpha=0.8,
)
plt.title('User Segments by Savings vs Non-Essential Ratio')
plt.tight_layout()
plt.show()